In [1]:
# Import core libraries
import pandas as pd
import numpy as np

In [3]:
# Import scikit-learn modules for Pipeline and Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib # For saving the pipeline

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
# 1. Load the Titanic dataset
df = pd.read_csv('../dataset/train.csv')

In [10]:
# Handle basic missing values to avoid pipeline errors early on
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

In [11]:
# 2. FEATURE ENGINEERING (Task Requirement: Create at least 2 new features)

# Feature 1: 'FamilySize' (Siblings/Spouses + Parents/Children + 1 for the passenger themselves)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Feature 2: 'IsAlone' (1 if FamilySize is 1, else 0)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Drop columns that are no longer needed or too complex for this baseline
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId', 'SibSp', 'Parch'], axis=1, inplace=True)

# Let's see the new features
df[['FamilySize', 'IsAlone', 'Survived']].head()

,FamilySize,IsAlone,Survived
0,2,0,0
1,2,0,1
2,1,1,1
3,2,0,1
4,1,1,0


In [12]:
# Separate features and target
X = df.drop('Survived', axis=1)
y = df['Survived']

# Split the dataset into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numerical and categorical columns for the pipeline
numerical_cols = ['Age', 'Fare', 'FamilySize', 'IsAlone']
categorical_cols = ['Sex', 'Pclass', 'Embarked'] 
# Note: Treating Pclass as categorical is often a good practice

In [13]:
# Create transformers for numerical and categorical data
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first')

# Bundle preprocessing for numerical and categorical data using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [14]:
# Define the final model (Using Random Forest as it works great with engineered features)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Bundle preprocessing and modeling code in a single Pipeline
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

# Preprocessing of training data, fit model (All happens in one step!)
my_pipeline.fit(X_train, y_train)

print("Pipeline successfully trained!")

Pipeline successfully trained!


In [15]:
# Preprocessing of validation data, get predictions
y_pred = my_pipeline.predict(X_test)

# Evaluate the model
print(f"Pipeline Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Note: The new features (FamilySize & IsAlone) combined with the RandomForest 
# typically push the accuracy higher than a basic Logistic Regression without them.

Pipeline Accuracy: 81.56%

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       105
           1       0.79      0.76      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.81      0.82      0.82       179



In [16]:
# Task Requirement: Save your final pipeline using joblib
pipeline_filename = 'titanic_rf_pipeline.joblib'
joblib.dump(my_pipeline, pipeline_filename)

print(f"Pipeline successfully saved as '{pipeline_filename}'.")
print("You can now load this file in any Python script to make predictions on new data.")

Pipeline successfully saved as 'titanic_rf_pipeline.joblib'.
You can now load this file in any Python script to make predictions on new data.
